In [0]:
dbutils.widgets.dropdown(name="environment", defaultValue="dev", choices=["dev", "qa", "prd"], label = "Select Environment")
env = dbutils.widgets.get("environment")
print(env)

catalog = f"saleslake_{env}"

bronzeTable = f"{catalog}.bronze_schema_{env}.rawStore"
silverTable = f"{catalog}.silver_schema_{env}.cleanedStore"
print(f"Source Table : {bronzeTable}")
print(f"Target Table : {silverTable}")

srcFileLoc=f"/Volumes/{catalog}/silver_{env}/vol_saleslake_src_files_{env}/daily_store/"
# print(srcFileLoc)

In [0]:
# Get Incremental Watermark
# This prevents loading already processed Bronze records.
watermark = spark.sql(f"""
SELECT
    COALESCE(
        MAX(ingest_ts),
        TIMESTAMP('1900-01-01 00:00:00')
    ) AS max_ingest_ts
FROM {silverTable}
""").collect()[0][0]

print("Current Watermark :", watermark)

In [0]:
# Transform and Load Store Data
spark.sql(f"""
INSERT INTO {silverTable}
SELECT DISTINCT
    CAST(TRIM(store_id) AS INT)               AS store_id,
    UPPER(TRIM(store_code))                   AS store_code,
    TRIM(store_name)                          AS store_name,
    UPPER(TRIM(store_type))                   AS store_type,
    TRIM(address)                             AS address,
    UPPER(TRIM(city))                         AS city,
    UPPER(TRIM(state))                        AS state,
    CAST(TRIM(region_id) AS INT)              AS region_id,
    TRIM(manager_name)                        AS manager_name,
    TO_DATE(TRIM(opening_date), 'yyyy-MM-dd') AS opening_date,
    CAST(TRIM(square_feet) AS INT)            AS square_feet,
    UPPER(TRIM(status))                       AS status,
    CURRENT_TIMESTAMP() AS ingest_ts
FROM {bronzeTable}
WHERE ingest_ts > (
    SELECT COALESCE(MAX(ingest_ts), TO_TIMESTAMP("1990-01-01"))
    FROM {silverTable}
)
ORDER BY store_id
""")
# WHERE ingest_ts > TIMESTAMP('{watermark}')
#     ORDER BY CAST(TRIM(store_id) AS INT)
#     """)

print(f"Silver load complete for {silverTable}")
print("Store Silver Load Completed Successfully")

In [0]:
spark.sql(f"SELECT COUNT(*) AS row_count FROM {silverTablName}").display()


In [0]:
# Store Record Count
spark.sql(f"""
SELECT
    COUNT(*) AS total_store_count
FROM {silverTable}
""").show()

In [0]:
# Duplicate Store Check
# Each store_id should appear only once in Silver.

spark.sql(f"""
SELECT
    store_id,
    COUNT(*) AS duplicate_count
FROM {silverTable}
GROUP BY store_id
HAVING COUNT(*) > 1
""").show()

In [0]:
# Null Store ID Check

spark.sql(f"""
SELECT
    COUNT(*) AS null_store_id
FROM {silverTable}
WHERE store_id IS NULL
""").show()

In [0]:
# Invalid Status Check
# Allowed statuses:
# ACTIVE
# CLOSED
# RENOVATION

spark.sql(f"""
SELECT
    store_id,
    store_name,
    status
FROM {silverTable}
WHERE status NOT IN (
    'ACTIVE',
    'CLOSED',
    'RENOVATION'
)
""").show()

In [0]:
# Invalid Square Feet Check
# Square feet should be positive.

spark.sql(f"""
SELECT
    store_id,
    square_feet
FROM {silverTable}
WHERE square_feet <= 0
OR square_feet IS NULL
""").show()

In [0]:
# Store Distribution by Status
# Business validation:

spark.sql(f"""
SELECT
    status,
    COUNT(*) AS store_count
FROM {silverTable}
GROUP BY status
ORDER BY store_count DESC
""").show()

In [0]:
# Sample Cleaned Data

spark.sql(f"""
SELECT *
FROM {silverTable}
ORDER BY store_id
LIMIT 10
""").show(truncate=False)